In [ ]:
import pandas as pd

In [ ]:
metadata_0_10 = pd.read_csv('/Users/madhu/Desktop/WFP/wfp_reports/wfp_metada_0_10.csv')
metadata_10_20 = pd.read_csv('/Users/madhu/Desktop/WFP/wfp_reports/wfp_metada_10_20.csv')
metadata_20_33 = pd.read_csv('/Users/madhu/Desktop/WFP/wfp_reports/wfp_metada_20_33.csv')
metadata_34_66 = pd.read_csv('/Users/madhu/Desktop/WFP/wfp_reports/wfp_metada_34_66.csv')
metadata_20 = pd.read_csv('/Users/madhu/Desktop/WFP/wfp_reports/wfp_metada_20.csv')

In [ ]:
reports_metadata = pd.concat([metadata_0_10, metadata_10_20, metadata_20_33, metadata_34_66, metadata_20]).drop_duplicates()

In [ ]:
reports_metadata.to_csv('/Users/madhu/Desktop/WFP/wfp_reports/reports_metadata.csv',index=False)

In [ ]:
reports_metadata['Topic'].value_counts()

In [ ]:
import os
import re
import requests
from pathlib import Path
from bs4 import BeautifulSoup

def download_pdfs_from_table(row):
    
    topic_name = row["Topic"]
    report_name = row["Report Name"]
    report_link = row["Report Link"]

    print(topic_name)

    base_download_folder = Path.home() / "Desktop" / "WFP" / "wfp_reports" / "wfp_reports"
    base_download_folder.mkdir(parents=True, exist_ok=True)

    clean_topic_name = re.sub(r"[^A-Za-z0-9_]+", "_", topic_name.split("(")[0]).strip("_")
    topic_folder = base_download_folder / clean_topic_name
    topic_folder.mkdir(parents=True, exist_ok=True)

    headers = {"User-Agent": "Mozilla/5.0"}
    records = []

    try:
        response = requests.get(report_link, headers=headers)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        table = soup.find("table", class_="wfp-wrapper wfp-table--data mb4")
        if not table:
            print(f"No table found for {report_link}")
            return pd.DataFrame([{
                "Topic": topic_name,
                "Report Name": report_name,
                "Report Link": report_link,
                "PDF Link": "No table found"
            }])

        rows = table.find_all("tr")[1:3]  # download first 2 rows max
        for tr in rows:
            cells = tr.find_all("td")
            if len(cells) < 4:
                continue

            sub_report_name = cells[0].get_text(strip=True)
            pdf_tag = tr.find("a", string="Download")
            pdf_url = pdf_tag["href"] if pdf_tag else None

            if pdf_url:
                pdf_filename = re.sub(r"[^\w\s-]", "", sub_report_name).replace(" ", "_") + ".pdf"
                pdf_path = topic_folder / pdf_filename

                pdf_resp = requests.get(pdf_url, headers=headers)
                if pdf_resp.ok:
                    with open(pdf_path, "wb") as f:
                        f.write(pdf_resp.content)
                    print(f"✅ Downloaded {pdf_filename}")
                else:
                    print(f"❌ Failed to download {pdf_url}")
            else:
                pdf_url = "No PDF found"

            records.append({
                "Topic": topic_name,
                "Report Name": sub_report_name,
                "Report Link": report_link,
                "PDF Link": pdf_url
            })

    except Exception as e:
        print(f"Error processing {report_link}: {e}")

    return pd.DataFrame(records)


In [ ]:
results_series = reports_metadata[reports_metadata['PDF Link'] == 'No PDF found'].sort_values(['Topic']) \
    .apply(download_pdfs_from_table, axis=1)

# Convert Series of DataFrames → single combined DataFrame
results_df = pd.concat(results_series.tolist(), ignore_index=True)


In [ ]:
workaround_report = reports_metadata[(reports_metadata['PDF Link'] == 'No PDF found')]
reports_metadata = reports_metadata[~(reports_metadata['PDF Link'] == 'No PDF found')]

In [ ]:
workaround_report.columns

In [ ]:
workaround_report = pd.merge(workaround_report.drop(['Report Name','PDF Link'],axis=1),results_df, on = ['Topic','Report Link'])

In [ ]:
workaround_report = workaround_report[workaround_report['PDF Link'] != 'No table found']

In [ ]:
final_metadata = pd.concat([workaround_report,reports_metadata])

In [ ]:
final_metadata.to_csv('/Users/madhu/Desktop/WFP/wfp_reports/reports_metadata.csv')